In [91]:
import pandas as pd
import os

In [92]:
DEST_FILE = "../data"
FILE_NAME = "TMDB_all_movies.csv"
full_path = os.path.join(DEST_FILE, FILE_NAME)

In [93]:
# Lecture du CSV
df = pd.read_csv(full_path)

In [94]:
# Tri
df_sorted = df.sort_values(
    by=["vote_count", "popularity", "vote_average"],
    ascending=[False, False, False]
)

In [95]:
# Garder seulement les 50000 premiers
df = df_sorted.head(50000).copy()

In [96]:
# Conversions numériques
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").astype(float)
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").astype("Int64")  # Int64 pour accepter NaN
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce").astype(float)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce").astype(float)
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce").astype(float)

In [97]:
# Dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

In [98]:
# Colonnes à transformer en listes
array_columns = [
    "genres", "production_countries", "production_companies", "cast", "director", "writers"
]

for col in array_columns:
    df[col + "_array"] = (
        df[col].str.split(r",\s*")       # découper sur virgule + espace
             .apply(lambda x: x if isinstance(x, list) else [])  # remplacer NaN par []
    )

# Suppression des colonnes originales
df = df.drop(columns=array_columns)

In [99]:
# Vérifier les types
print(df.dtypes)

id                                     int64
title                                 object
vote_average                         float64
vote_count                             Int64
status                                object
release_date                  datetime64[ns]
revenue                              float64
runtime                              float64
budget                               float64
imdb_id                               object
original_language                     object
original_title                        object
overview                              object
popularity                           float64
tagline                               object
spoken_languages                      object
director_of_photography               object
producers                             object
music_composer                        object
imdb_rating                          float64
imdb_votes                           float64
poster_path                           object
release_ye

In [100]:
df = df.drop(
    ["status", "imdb_id", "tagline", "director_of_photography",
     "producers", "imdb_rating", "imdb_votes",
     "music_composer", "revenue", "spoken_languages", "original_language"],
    axis=1
)

In [101]:
# Delete line with empty title
df = df[df["title"].notna() & (df["title"] != "")]

In [102]:
# Delete line with empty overview
df = df[df["overview"].notna() & (df["overview"] != "")]

In [103]:
df = df.fillna({
    'release_year': -1
})

In [104]:
#  ne garder que les lignes qui ont un runtime supérieur à 44.0
df = df[df["runtime"] > 44.0]

In [105]:
# enlever les films qui ont uniquement la valeur 'Documentary' dans la colonne genres
df = df[~df["genres_array"].apply(lambda x: len(x) == 1 and x[0] == "Documentary")]

In [106]:
df.shape

(45355, 18)

In [107]:
def count_empty_values(df):
    counts = {}
    for col in df.columns:
        counts[col] = (
            df[col].isna()                                  # NaN / None
            | (df[col] == "")                               # chaîne vide
            | (df[col].apply(lambda x: isinstance(x, list) and len(x) == 0))  # liste vide
        ).sum()
    return pd.Series(counts, name="empty_count")

In [108]:
empty_counts = count_empty_values(df)
print(empty_counts)

id                               0
title                            0
vote_average                     0
vote_count                       0
release_date                     2
runtime                          0
budget                           0
original_title                   0
overview                         0
popularity                       0
poster_path                     78
release_year                     0
genres_array                    31
production_countries_array     568
production_companies_array    1856
cast_array                      50
director_array                  78
writers_array                 1009
Name: empty_count, dtype: int64


In [109]:
clean_path = os.path.join(DEST_FILE, "TMDB_clean.csv")

In [110]:
if os.path.exists(clean_path):
    os.remove(clean_path)

In [111]:
df.to_csv(clean_path, index=False)

## ML

In [112]:
from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
# from scipy.sparse import hstack
import umap.umap_ as umap
import numpy as np

In [113]:
# Vectorisation TF-IDF sur les résumés
# faire des tests de temps en temps avec max_features=1000
vectorizer = TfidfVectorizer(max_features=1000, stop_words="english")
X_tfidf = vectorizer.fit_transform(df["overview"])

In [128]:
# Réduction dimensionnelle avec UMAP
# reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric="cosine",
    init="spectral",
    random_state=42
)
embedding = reducer.fit_transform(X_tfidf.toarray())

df["x"] = embedding[:,0]
df["y"] = embedding[:,1]

/usr/local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [115]:
# def top_k_binarize(series, top_k=100):
#     # Compte la fréquence de chaque item
#     all_items = [item for sublist in series for item in sublist]
#     uniques, counts = np.unique(all_items, return_counts=True)
#     top_items = set([u for u, c in sorted(zip(uniques, counts), key=lambda x: -x[1])[:top_k]])

#     # Filtre et binarize
#     filtered_series = [[i for i in lst if i in top_items] for lst in series]
#     mlb = MultiLabelBinarizer(classes=list(top_items))
#     return mlb.fit_transform(filtered_series)

In [116]:
# genres_matrix = top_k_binarize(df['genres_array'], top_k=50)
# cast_matrix = top_k_binarize(df['cast_array'], top_k=500)
# director_matrix = top_k_binarize(df['director_array'], top_k=200)
# writers_matrix = top_k_binarize(df['writers_array'], top_k=200)
# production_companies_matrix = top_k_binarize(df['production_companies_array'], top_k=100)
# production_countries_matrix = top_k_binarize(df['production_countries_array'], top_k=50)

In [117]:
# scaler = StandardScaler()
# year_matrix = scaler.fit_transform(df[['release_year']])

In [118]:
# # Concaténer tout dans un espace euclidien
# features = np.hstack([
#     embedding * 2.0,                # TF-IDF / UMAP de l’overview → très important pour le contenu
#     genres_matrix * 1.5,            # Genres → assez important, mais moins que le texte
#     production_countries_matrix * 0.3,  # Pays → influence plus faible
#     production_companies_matrix * 0.3,  # Société de prod → faible
#     cast_matrix * 0.8,              # Cast → modérément important
#     director_matrix * 1.0,          # Réalisateur → important
#     writers_matrix * 0.5,           # Scénaristes → moins important que réalisateur/cast
#     year_matrix * 0.2               # Année de sortie → faible importance
# ])

In [119]:
def find_closest_movies(df, movie_id, top_n=10):
    if movie_id not in df['id'].values:
        print(f"Le film '{movie_id}' n'a pas été trouvé.")
        return None
    
    input_point = df.loc[df['id'] == movie_id, ['x','y']].values[0]
    df['distance'] = np.linalg.norm(df[['x','y']].values - input_point, axis=1)
    
    df_filtered = df[df['id'] != movie_id]
    closest_movies = df_filtered.nsmallest(top_n, 'distance')
    
    return closest_movies[['id', 'title', 'vote_average', 'vote_count', 'release_date', 'runtime', 'overview', 'popularity', 'poster_path', 'genres_array', 'production_countries_array', 'production_companies_array', 'cast_array', 'director_array', 'writers_array', 'distance']]

In [130]:
closest_10 = find_closest_movies(df, 424, top_n=10)
print(closest_10)

            id                                   title  vote_average  \
704916  999125                                   Filip         6.737   
509        637                       Life Is Beautiful         8.443   
4427      9075                              Black Book         7.400   
42354    61623                 Jew Suss: Rise and Fall         5.308   
326379  477428                        The Painted Bird         7.129   
275399  412467  Beyond Valkyrie: Dawn of the 4th Reich         5.477   
335478  489176                         The Birdcatcher         5.532   
38484    56179                   Triumph of the Spirit         6.600   
25382    40029               The Bamboo House of Dolls         5.231   
3878      7862                      The Counterfeiters         7.363   

        vote_count release_date  runtime  \
704916          38   2023-03-03    119.0   
509          13473   1997-12-20    116.0   
4427          1184   2006-09-14    145.0   
42354           26   2010-09-23

In [121]:
# def recommend_hybrid(df, input_title, features, top_n=10):
#     if input_title not in df['title'].values:
#         print(f"Le film '{input_title}' n'a pas été trouvé.")
#         return None
    
#     # Coordonnées du film
#     input_idx = df[df['title'] == input_title].index[0]
#     input_vector = features[input_idx]

#     # Distances euclidiennes
#     distances = np.linalg.norm(features - input_vector, axis=1)

#     # Exclure le film lui-même
#     df['distance'] = distances
#     df_filtered = df[df['title'] != input_title]

#     # Récupérer les top_n les plus proches
#     closest = df_filtered.nsmallest(top_n, 'distance')
#     return closest[['title','distance']]

In [122]:
# closest_movies = recommend_hybrid(df, "Schindler's List", features, top_n=10)
# print(closest_movies)